In [ ]:
# 다음리뷰데이터 로드
# 평점 최저 최고 점수 의 절반을 threshold로 정하고 0(긍정) 1(부정)로 추가 컬럼 생성

- 단어를 밀집 백터(Dense Vector)로 변환해서 모델이 단어간의 의미적 유사성을 직접학습
- 단어의 순서 정보를 활용할수 있음
- 학습데이터가 많을수록 단어간의 관계를 더 깊게 파악

- 단점
- 모든문장의 길이를 맞추기위해서 패딩과정이필요 padding
```
- 단어사전관리(Vocabulary) 특수토큰 ( <PAD> ,<UNK> )
```


In [ ]:
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
df.head()

In [ ]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, _ in okt.pos(text, stem=True)]

tfidf = TfidfVectorizer(tokenizer=kor_tokenize,max_features=5000)
x_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['target'].values

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
class TfidfDataset(Dataset):
    def __init__(self,vectors, labels):
        self.vectors = torch.FloatTensor(vectors)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.vectors[index], self.labels[index]

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_tfidf,y,random_state=42,test_size=0.2)
train_loader = DataLoader(TfidfDataset(x_train,y_train), batch_size=32,shuffle=True)
test_loader = DataLoader(TfidfDataset(x_test, y_test), batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
class TfidfMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim,1),
        )
    def forward(self, x):
        return self.network(x)    

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from torch.optim import Adam
x_train.shape[1]
model = TfidfMLP(input_dim = x_train.shape[1], hidden_dim = 64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [ ]:
%pip install ipywidgets

In [ ]:
from tqdm import tqdm
epochs = 10
pbar = tqdm(range(epochs), desc="Training")
for epoch in pbar:
    model.train()
    local_loss = 0.0
    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(vecs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    avg_loss = local_loss / len(train_loader)
    pbar.set_postfix(
        loss=f"{avg_loss:.4f}"
    )   

In [ ]:
# 예측... 정확도
model.eval()
correct  = 0.0; total = 0.0
with torch.no_grad():
    for vecs, labels in test_loader:
        vecs,labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
    print(f'test accuracy : {(correct / total):.4f}')